<a href="https://colab.research.google.com/github/ritterb64/NeoCredit_LimpiezaDatos/blob/main/NeoCredit_LimpizaDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##LIMPIEZA DE DATOS NeoCredit SA
##Profesora: Adriana Collaguazo
##Integrantes:

*   Ritter Briones
*   Mariana Mora
*   Anthony Polo

Paso 1: Preparación del entorno de trabajo

In [1]:
#instalacion de librearia pandera
!pip install pandera ydata-profiling missingno -q

#importar librerias
import pandas as pd
import numpy as np
import missingno as msno
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import(
    MinMaxScaler
    ,RobustScaler
    ,MaxAbsScaler
    ,StandardScaler
    ,PowerTransformer
    ,Normalizer)
import pandera.pandas as pa
from pandera import  Column,Check,DataFrameSchema

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.8 MB/s eta 0:00:00


In [2]:
#lectura de dataset
df=pd.read_csv("/content/solicitudes_credito_neocredit.csv")

Paso 2: Exploración inicial y perfilado de datos

In [3]:
#inspeccion de tipo de datos
df.info()
#el data frame cuenta con con 669  registros
#se observa que dispositivos,fecha_solicitud, genero y canal_solicitud son las columna con mayor cantidad de datos faltantes
# existen columnas object que deberian ser numericos como ingreso_mensual

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id_solicitud               669 non-null    object 
 1   id_cliente                 663 non-null    object 
 2   nombre_cliente             663 non-null    object 
 3   edad                       663 non-null    float64
 4   genero                     578 non-null    object 
 5   ciudad                     614 non-null    object 
 6   ingreso_mensual            642 non-null    object 
 7   tipo_empleo                607 non-null    object 
 8   antiguedad_laboral_meses   663 non-null    float64
 9   monto_solicitado           663 non-null    float64
 10  plazo_meses                663 non-null    float64
 11  score_buro_externo         635 non-null    float64
 12  deuda_actual               663 non-null    float64
 13  num_tarjetas_activas       663 non-null    float64

In [4]:
#estadistica descriptiva
df.describe(include="all")
#se observan valores atipicos en edad donde el minimo valor es -5 y maximo 210
#se observan valores atipicos en antiguedad_laboral_meses donde el minimo valor es -12

,id_solicitud,id_cliente,nombre_cliente,edad,genero,ciudad,ingreso_mensual,tipo_empleo,antiguedad_laboral_meses,monto_solicitado,...,score_buro_externo,deuda_actual,num_tarjetas_activas,historial_pagos_atrasados,fecha_solicitud,canal_solicitud,dispositivo,ip_pais,resultado_solicitud,fraude_flag
count,669,663,663,663.000000,578,614,642,607,663.000000,663.000000,...,635.000000,663.000000,663.000000,663.000000,570,572,563,663,663,663.000000
unique,656,529,560,NaN,7,12,610,8,NaN,NaN,...,NaN,NaN,NaN,NaN,505,7,6,7,5,NaN
top,SOL-10630,CLI-1215,Ambrosio Barrera,NaN,F,Cuenca,No disponible,independiente,NaN,NaN,...,NaN,NaN,NaN,NaN,2025-12-16,Call Center,iOS,Ecuador,Pendiente,NaN
freq,2,4,4,NaN,94,62,20,85,NaN,NaN,...,NaN,NaN,NaN,NaN,3,99,103,565,151,NaN
mean,NaN,NaN,NaN,46.586727,NaN,NaN,NaN,NaN,113.235294,3855.039653,...,564.193701,7417.754419,2.963801,1.164404,NaN,NaN,NaN,NaN,NaN,0.046757
std,NaN,NaN,NaN,18.836343,NaN,NaN,NaN,NaN,69.470410,3465.585273,...,183.382834,4385.976062,1.946851,1.161492,NaN,NaN,NaN,NaN,NaN,0.211278
min,NaN,NaN,NaN,-5.000000,NaN,NaN,NaN,NaN,-12.000000,451.930000,...,-1.000000,19.220000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,32.000000,NaN,NaN,NaN,NaN,56.500000,1040.040000,...,431.000000,3478.785000,1.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000
50%,NaN,NaN,NaN,47.000000,NaN,NaN,NaN,NaN,109.000000,2377.660000,...,575.000000,7584.650000,3.000000,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000
75%,NaN,NaN,NaN,61.000000,NaN,NaN,NaN,NaN,173.000000,5717.945000,...,711.000000,11234.710000,5.000000,2.000000,NaN,NaN,NaN,NaN,NaN,0.000000
